# Sentinel Airline Volatility Model: EGARCH vs TimesFM-3 (+ HMM regimes, PCA)

**What this notebook does:** forecasts how volatile each of 8 airline stocks will be over the **next ~2 months**
(42 trading days) and saves everything the Sentinel website needs into one file, `sentinel_data.json`.

**Steps**
1. Download 8 airlines and compute returns
2. HMM finds the market regime (Calm / Stressed / Crisis) from the sector's returns
3. Forecast the next 2 months with **EGARCH** (a classic statistical model)
4. Forecast the next 2 months with **TimesFM-3** (Google's pre-trained AI forecasting model), two versions
5. PCA: how much of each airline's risk is sector-wide vs stock-specific
6. Final table: one forecast per airline
7. Export `sentinel_data.json` for the website

There is no backtest here. Nothing is scored against what happened; these are forward-looking forecasts only.

In [ ]:
%pip install -q yfinance pandas numpy arch hmmlearn scikit-learn matplotlib

# TimesFM-3 (installs from Google's GitHub; weights download automatically on first use)
!git clone -q https://github.com/google-research/timesfm.git
%pip install -q -e "./timesfm[torch]"

import glob
import json
import math
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.decomposition import PCA
import torch

warnings.filterwarnings('ignore')

## 1. Settings (change things here)

In [ ]:
TICKERS = ['DAL', 'AAL', 'UAL', 'LUV', 'ALK', 'JBLU', 'ULCC', 'ALGT']

START = '2018-01-01'   # includes COVID so the models see a real crisis

HORIZON = 42           # forecast this many trading days ahead (about 2 months: 21 per month)
N_SIMS = 300           # simulations used for multi-day EGARCH forecasts
PCA_WINDOW = 120       # days of data per PCA window
TRADING_DAYS = 252     # used to annualize volatility
VOL_WINDOW = 20        # TimesFM forecasts a rolling volatility series built from this many days
CONTEXT = 512          # how many days of history TimesFM gets to look at
N_HIST = 252           # how much history (about 1 year) goes into the website charts

REGIMES = ['Calm', 'Stressed', 'Crisis']

## 2. Data
Returns are log returns **x100** (so they are in %). The `arch` library fits more reliably this way.

ULCC (Frontier) only started trading in 2021, so it has no data before then. That is fine: each airline uses the data it has.
The **sector return** is the average return of all airlines trading that day. The HMM uses it to detect regimes.

We also keep **volume** for the website's price charts.

In [ ]:
raw = yf.download(TICKERS, start=START, progress=False)
prices = raw['Close'][TICKERS]
volume = raw['Volume'][TICKERS]

returns = np.log(prices / prices.shift(1)) * 100
returns = returns.iloc[1:].dropna(how='all')

sector = returns.mean(axis=1)   # average airline return each day

print(returns.index[0].date(), 'to', returns.index[-1].date(), '|', len(returns), 'days')
print('First available date per airline:')
print(returns.apply(lambda s: s.first_valid_index().date()))

## 3. HMM: which regime are we in?
The HMM looks at the sector returns and sorts days into 3 hidden states. We order the states by volatility:
lowest = **Calm**, middle = **Stressed**, highest = **Crisis**.

It outputs, for any day, the **probability** of being in each regime. This is shown on the website's Portfolio page.

In [ ]:
def fit_hmm(sector_returns):
    '''Fit a 3-state HMM and return it with the state order (calm -> crisis).'''
    X = sector_returns.values.reshape(-1, 1)
    hmm = GaussianHMM(n_components=3, covariance_type='full',
                      n_iter=200, random_state=0).fit(X)
    order = np.argsort(hmm.covars_.flatten())   # smallest variance first
    return hmm, order


def get_regimes(hmm, order, sector_returns):
    '''
    Return:
      probs  - daily probability of each regime (Calm, Stressed, Crisis)
      labels - the most likely regime each day (0, 1, 2)
      trans  - probability of moving from one regime to another
    '''
    X = sector_returns.values.reshape(-1, 1)
    probs = hmm.predict_proba(X)[:, order]
    labels = probs.argmax(axis=1)
    trans = hmm.transmat_[np.ix_(order, order)]
    return probs, labels, trans


hmm, order = fit_hmm(sector)
probs, labels, trans = get_regimes(hmm, order, sector)

summary = pd.DataFrame({
    'Days': [(labels == k).sum() for k in range(3)],
    'Daily vol (%)': [sector.values[labels == k].std() for k in range(3)],
}, index=REGIMES)
print(summary.round(2))
print('\nChance of tomorrow\'s regime, given today\'s (rows = today):')
print(pd.DataFrame(trans, index=REGIMES, columns=REGIMES).round(3))
print('\nCurrent regime probabilities:', dict(zip(REGIMES, probs[-1].round(3))))

In [ ]:
# Plot regime probabilities over time
p = pd.DataFrame(probs, index=sector.index, columns=REGIMES)
p.plot.area(figsize=(14, 3.5), color=['green', 'orange', 'red'], alpha=0.7)
plt.title('HMM regime probabilities')
plt.ylabel('Probability')
plt.show()

## 4. EGARCH forecast
One EGARCH model per airline, fit on all of its returns. It forecasts the volatility for each of the next `HORIZON` trading days.

*Note: `o=1` adds the asymmetry term, so bad-news days raise volatility more than good-news days. Without it, EGARCH ignores asymmetry.*

**Safety check:** on some stocks the `arch` library's multi-day forecast can come out near zero even though the fit looks fine
(this showed up on a short, noisy history). So each forecast's first day is checked against a direct calculation from the fitted
parameters. If they disagree, a backup simulation of the same EGARCH equation is used instead, and the airline is listed.

In [ ]:
def make_model(r):
    return arch_model(r, vol='EGARCH', p=1, o=1, q=1)


def variance_path(fitted):
    '''Forecast the daily variance for each of the next HORIZON days.'''
    f = fitted.forecast(horizon=HORIZON, method='simulation',
                        simulations=N_SIMS, reindex=False)
    return f.variance.values[-1]


def to_annual_vol(var_path):
    '''Average variance over the horizon -> annualized volatility (%).'''
    return np.sqrt(var_path.mean()) * np.sqrt(TRADING_DAYS)


def own_variance_path(fitted):
    '''Backup: simulate the EGARCH equation directly from the fitted parameters.'''
    p = fitted.params
    omega, alpha, gamma, beta = p['omega'], p['alpha[1]'], p['gamma[1]'], p['beta[1]']
    rng = np.random.default_rng(0)
    log_var = np.full(N_SIMS, np.log(fitted.conditional_volatility.iloc[-1] ** 2))
    z = np.full(N_SIMS, fitted.std_resid.iloc[-1])      # last standardized shock
    path = []
    for _ in range(HORIZON):
        log_var = omega + alpha * (np.abs(z) - np.sqrt(2 / np.pi)) + gamma * z + beta * log_var
        path.append(np.exp(log_var).mean())
        z = rng.standard_normal(N_SIMS)
    return np.array(path)


def checked_variance_path(fitted):
    '''arch's forecast, unless its first day disagrees with the direct calculation.'''
    path = variance_path(fitted)
    backup = own_variance_path(fitted)
    ratio = path[0] / backup[0]
    if (not np.isfinite(path).all()) or ratio < 0.8 or ratio > 1.25:
        return backup, True
    return path, False


egarch_var = {}    # airline -> daily variance path
used_backup = []
for t in TICKERS:
    r = returns[t].dropna()
    fitted = make_model(r).fit(disp='off')
    egarch_var[t], flagged = checked_variance_path(fitted)
    if flagged:
        used_backup.append(t)
if used_backup:
    print('Backup EGARCH simulation used for:', used_backup)

# Daily volatility path, annualized (%), the same units as the TimesFM paths below
egarch_path = {t: np.sqrt(v) * np.sqrt(TRADING_DAYS) for t, v in egarch_var.items()}
egarch_avg = {t: to_annual_vol(v) for t, v in egarch_var.items()}
print({t: round(v, 1) for t, v in egarch_avg.items()})

## 5. TimesFM-3: an AI forecasting model
TimesFM-3 is a pre-trained model from Google Research. It was trained on over a trillion data points from many fields, so we don't train it. We just give it a history and it forecasts.

EGARCH works from *returns*. TimesFM works directly on a *volatility series*, so we first build one: each day's volatility over the last 20 days (annualized).

Two versions:
- **TimesFM (alone):** each airline's volatility series is forecast on its own.
- **TimesFM (together):** all 8 airlines are forecast jointly, with the sector's volatility given as extra information.

Each forecast is a path of daily volatility for the next `HORIZON` days.

*License note: the TimesFM-3 weights are free for research and non-commercial use only.*

In [ ]:
# Volatility series built from past returns only
vol_df = returns.rolling(VOL_WINDOW).std() * np.sqrt(TRADING_DAYS)
sector_vol = sector.rolling(VOL_WINDOW).std() * np.sqrt(TRADING_DAYS)

# Load TimesFM-3 (downloads the model the first time)
from timesfm3 import TimesFM3Evaluator, ModelConfig

tfm = TimesFM3Evaluator(ModelConfig(
    checkpoint_path='google/timesfm-3.0-pytorch',
    per_core_batch_size=16,
    device='cuda' if torch.cuda.is_available() else 'cpu'))


def timesfm_paths():
    '''
    Forecast the next HORIZON days of volatility for all airlines.
    Returns two arrays of shape (8 airlines, HORIZON): alone, together.
    '''
    hist = vol_df.dropna().iloc[-CONTEXT:]                  # all 8 airlines, last CONTEXT days
    target = hist.values.T.astype(np.float32)               # shape: (8 airlines, days)
    sector_cov = sector_vol.reindex(hist.index).values[None, :].astype(np.float32)

    # Version 1: each airline on its own
    out_alone = list(tfm.predict_batch([target], horizon=HORIZON, univariate=True,
                                       return_quantiles=True,
                                       use_symmetric_averaging=False))[0]

    # Version 2: all airlines together, plus sector volatility as extra information
    out_together = list(tfm.predict_batch([target], horizon=HORIZON,
                                          past_only_covariates=[sector_cov],
                                          return_quantiles=True,
                                          use_symmetric_averaging=False))[0]

    return np.asarray(out_alone.forecast), np.asarray(out_together.forecast)


tfm_alone_path, tfm_together_path = timesfm_paths()
tfm_alone_avg = dict(zip(TICKERS, tfm_alone_path.mean(axis=1)))
tfm_together_avg = dict(zip(TICKERS, tfm_together_path.mean(axis=1)))

# Business days covered by the forecast
future_dates = pd.bdate_range(returns.index[-1] + pd.Timedelta(days=1), periods=HORIZON)
print('Forecast window:', future_dates[0].date(), 'to', future_dates[-1].date())

In [ ]:
# Recent volatility (what already happened) next to each model's forecast, per airline
fig, axes = plt.subplots(4, 2, figsize=(14, 14))
for j, (ax, t) in enumerate(zip(axes.flatten(), TICKERS)):
    recent = vol_df[t].dropna().tail(120)
    ax.plot(recent.index, recent.values, color='black', label='Recent volatility (20-day)')
    ax.plot(future_dates, egarch_path[t], color='blue', linestyle='--', label='EGARCH')
    ax.plot(future_dates, tfm_alone_path[j], color='green', linestyle=':', label='TimesFM (alone)')
    ax.plot(future_dates, tfm_together_path[j], color='purple', linestyle=':', label='TimesFM (together)')
    ax.set_title(t)
    ax.set_ylabel('Annualized vol (%)')
axes[0, 0].legend()
plt.tight_layout()
plt.show()

## 6. PCA: sector risk vs stock-specific risk
PCA finds the main pattern that all airlines share. The first component is the **sector factor**.

For each airline: **sector share** = % of its variance explained by that shared factor. The rest is **stock-specific**.
The window is rolling (`PCA_WINDOW` days) so we can see whether the sector share changes over time.
This feeds the website's Portfolio page.

In [ ]:
def sector_share(window):
    '''Share of each airline's variance explained by the sector factor (0 to 1).'''
    z = (window - window.mean()) / window.std()     # put all airlines on the same scale
    pca = PCA(n_components=1).fit(z)
    share = pca.components_[0] ** 2 * pca.explained_variance_[0]
    return pd.Series(share, index=window.columns)


aligned = returns.dropna()      # days where all 8 airlines have data

pca_rows = {}
for end in range(PCA_WINDOW, len(aligned) + 1, 5):
    window = aligned.iloc[end - PCA_WINDOW: end]
    pca_rows[aligned.index[end - 1]] = sector_share(window)

pca_history = pd.DataFrame(pca_rows).T
latest_share = sector_share(aligned.iloc[-PCA_WINDOW:])

pca_history.plot(figsize=(14, 4), alpha=0.7)
plt.title('Share of risk explained by the sector (rolling)')
plt.ylabel('Sector share')
plt.show()

print('Latest sector share per airline:')
print((latest_share * 100).round(1).astype(str) + '%')

## 7. Final output: one forecast per airline (next ~2 months)
Average annualized volatility over the next `HORIZON` trading days, from each model.
Volatility adds up in squares, so the EGARCH number is split into:
- sector vol = vol x sqrt(sector share)
- stock-specific vol = vol x sqrt(1 - sector share)

In [ ]:
final = []
for j, t in enumerate(TICKERS):
    vol = egarch_avg[t]
    share = latest_share[t]
    final.append({'Airline': t,
                  'EGARCH %': vol,
                  'TimesFM alone %': tfm_alone_avg[t],
                  'TimesFM together %': tfm_together_avg[t],
                  'Sector share': share,
                  'Sector vol % (EGARCH)': vol * np.sqrt(share),
                  'Stock-specific vol % (EGARCH)': vol * np.sqrt(1 - share)})

final = pd.DataFrame(final).set_index('Airline')
final.round(2).sort_values('EGARCH %', ascending=False)

## 8. Export for the website
Creates **`sentinel_data.json`** with everything the site shows: prices and volume, the 2-month forecast paths,
company info (bid, ask, PEG and so on), regime probabilities, and the correlation matrix and PCA for the Portfolio page.

**News, delay and fuel scores** come from your `combine.py` output. If a file named `composite_<date>.csv` is
in this Colab session (drag it into the Files panel on the left, or put it in an `output/` folder),
it is read and added to the JSON. If none is found, the export still works without it.

Fields Yahoo does not have (for example PEG for airlines with negative earnings) are saved as empty and the site shows N/A.

In [ ]:
def num(x, d=4):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return round(x, d) if math.isfinite(x) else None


def nums(seq, d=4):
    return [num(v, d) for v in seq]


def day_strs(idx):
    return [pd.Timestamp(d).strftime('%Y-%m-%d') for d in idx]


def cell_value(v):
    '''Turn one CSV cell into something JSON-safe: numbers stay numbers, the rest become text.'''
    if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool):
        return num(v, 4)
    if pd.isna(v):
        return None
    return str(v)


out = {
    'generated': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'horizon_days': HORIZON,
    'horizon_label': 'next ~2 months',
    'tickers': TICKERS,
    'forecast_dates': day_strs(future_dates),
    'stocks': {},
}

INFO_KEYS = ['shortName', 'longName', 'longBusinessSummary', 'sector', 'industry', 'website',
             'fullTimeEmployees', 'marketCap', 'currentPrice', 'regularMarketPrice',
             'bid', 'ask', 'trailingPE', 'forwardPE', 'pegRatio', 'trailingPegRatio',
             'fiftyTwoWeekHigh', 'fiftyTwoWeekLow', 'beta', 'averageVolume']

for j, t in enumerate(TICKERS):
    px = prices[t].dropna()
    ma50 = px.rolling(50).mean()
    ma200 = px.rolling(200).mean()
    px_tail = px.tail(N_HIST)
    vol_tail = volume[t].reindex(px_tail.index)

    try:
        info = yf.Ticker(t).info
    except Exception:
        info = {}
    snap = {k: info.get(k) for k in INFO_KEYS}

    realized = vol_df[t].dropna().tail(N_HIST)
    out['stocks'][t] = {
        'info': snap,
        'price': {
            'dates': day_strs(px_tail.index),
            'close': nums(px_tail.values, 2),
            'ma50': nums(ma50.reindex(px_tail.index).values, 2),
            'ma200': nums(ma200.reindex(px_tail.index).values, 2),
            'volume': nums(vol_tail.values, 0),
        },
        'realized_vol': {'dates': day_strs(realized.index), 'values': nums(realized.values, 2)},
        'egarch_path': nums(egarch_path[t], 2),
        'timesfm_alone_path': nums(tfm_alone_path[j], 2),
        'timesfm_together_path': nums(tfm_together_path[j], 2),
        'forecast_avg': {
            'egarch': num(egarch_avg[t], 2),
            'timesfm_alone': num(tfm_alone_avg[t], 2),
            'timesfm_together': num(tfm_together_avg[t], 2),
        },
    }

# ---- regimes (sector level; shown on the Portfolio page) ----
p_tail = pd.DataFrame(probs, index=sector.index, columns=REGIMES).tail(N_HIST)
out['regimes'] = {
    'dates': day_strs(p_tail.index),
    'Calm': nums(p_tail['Calm'].values, 3),
    'Stressed': nums(p_tail['Stressed'].values, 3),
    'Crisis': nums(p_tail['Crisis'].values, 3),
    'current': {k: num(v, 3) for k, v in zip(REGIMES, probs[-1])},
    'transition': [nums(row, 3) for row in trans],
}

# ---- portfolio page: returns, correlation, PCA ----
win = aligned.tail(PCA_WINDOW)
corr = win.corr()
z = (win - win.mean()) / win.std()
pca3 = PCA(n_components=min(3, len(TICKERS))).fit(z)
out['portfolio'] = {
    'returns_dates': day_strs(aligned.tail(N_HIST).index),
    'returns': {t: nums(aligned[t].tail(N_HIST).values, 3) for t in TICKERS},
    'correlation': {t: nums(corr[t].values, 3) for t in TICKERS},
    'pca_explained': nums(pca3.explained_variance_ratio_, 3),
    'pca_loadings': {t: nums(pca3.components_[:, k], 3) for k, t in enumerate(TICKERS)},
    'sector_share_latest': {t: num(latest_share[t], 3) for t in TICKERS},
    'sector_share_history': {
        'dates': day_strs(pca_history.index),
        **{t: nums(pca_history[t].values, 3) for t in TICKERS},
    },
}

# ---- news / delays / fuel composite from combine.py (optional) ----
comp_files = sorted(glob.glob('composite_*.csv') + glob.glob('output/composite_*.csv'))
if comp_files:
    comp = pd.read_csv(comp_files[-1], index_col=0)
    out['composite_file'] = os.path.basename(comp_files[-1])
    out['composite'] = {t: {c: cell_value(comp.loc[t, c]) for c in comp.columns}
                        for t in TICKERS if t in comp.index}
    print('Added composite scores from', comp_files[-1])
else:
    out['composite'] = {}
    print('No composite_<date>.csv found: exporting without news/delay/fuel scores.')
    print('Drag one into the Files panel (left side) and re-run this cell to include it.')

# ---- save + download ----
with open('sentinel_data.json', 'w') as f:
    json.dump(out, f, default=str)
final.round(3).to_csv('final_forecasts.csv')
print('Wrote sentinel_data.json  (%.0f KB)' % (os.path.getsize('sentinel_data.json') / 1024))

try:
    from google.colab import files
    files.download('sentinel_data.json')
    files.download('final_forecasts.csv')
except ImportError:
    print('Not running in Colab: files are saved in the current folder.')